# 知网网课挂机助手 · 成果展示

本 notebook 展示 `刷网课.py` 的核心逻辑与运行结果，**可复现、不含真实账号密码**。

> 使用：`Kernel → Restart & Run All` 或逐个运行单元格。

In [1]:
import sys, platform
print("Python:", sys.version.split()[0])
print("OS:", platform.system(), platform.release())
try:
    import playwright
    print("playwright: 已安装（可选依赖，运行主程序需安装）")
except Exception:
    print("playwright: 未安装（本 demo 不需要，主程序需要）")


Python: 3.11.5
OS: Windows 10
playwright: 已安装（可选依赖，运行主程序需安装）


## 1) 核心思路：绕过"必须在台前"的检测

网课常要求浏览器保持前台才计时。工具通过三招解决：

In [1]:
# 与 刷网课.py 一致的核心常驻参数
INIT_SCRIPT = r"""
(() => {
  const vis = () => 'visible';
  try { Object.defineProperty(document, 'visibilityState', { get: vis, configurable: true }); } catch (e) {}
  try { Object.defineProperty(document, 'webkitVisibilityState', { get: vis, configurable: true }); } catch (e) {}
  for (const k of ['hidden', 'webkitHidden', 'mozHidden', 'msHidden']) {
    try { Object.defineProperty(document, k, { get: () => false, configurable: true }); } catch (e) {}
  }
  try { Object.defineProperty(document, 'hasFocus', { value: () => true, configurable: true }); } catch (e) {}
  try { Object.defineProperty(window, 'onblur', { value: null, configurable: true }); } catch (e) {}
})();
"""
PLAYER_FLAGS = ['--mute-audio', '--autoplay-policy=no-user-gesture-required', '--disable-background-timer-throttling', '--disable-backgrounding-occluded-windows', '--disable-renderer-backgrounding', '--disable-features=CalculateNativeWinOcclusion', '--no-default-browser-check', '--disable-popup-blocking']

print("PLAYER_FLAGS:")
for f in PLAYER_FLAGS:
    print("   ", f)
print()
print("INIT_SCRIPT 前 3 行:")
print("\n".join(INIT_SCRIPT.strip().splitlines()[:3]))


PLAYER_FLAGS:
    --mute-audio
    --autoplay-policy=no-user-gesture-required
    --disable-background-timer-throttling
    --disable-backgrounding-occluded-windows
    --disable-renderer-backgrounding
    --disable-features=CalculateNativeWinOcclusion
    --no-default-browser-check
    --disable-popup-blocking

INIT_SCRIPT 前 3 行:
(() => {
  const vis = () => 'visible';
  try { Object.defineProperty(document, 'visibilityState', { get: vis, configurable: true }); } catch (e) {}


## 2) 账号密码读取（不硬编码、已打码）

In [1]:
import re

# 与 刷网课.py 的 read_creds 逻辑一致：正则解析 账号/密码，不硬编码
def read_creds(text):
    um = re.search(r"账号[:：]\s*(\S+)", text)
    pm = re.search(r"密码[:：]\s*(\S+)", text)
    return (um.group(1) if um else ""), (pm.group(1) if pm else "")

def mask(s):
    return (s[:2] + "***") if s else "(空)"

t = "账号：示例账号\n密码：示例密码"
u, p = read_creds(t)
print("账号:", mask(u), "  密码: 已读取, 长度 =", len(p))
print("实际运行时从 账号密码.md 读取；本示例已打码，不泄露真实值。")


账号: 示例***   密码: 已读取, 长度 = 4
实际运行时从 账号密码.md 读取；本示例已打码，不泄露真实值。


## 3) 进度判定与统计（以服务端为准）

In [1]:
def fmt_hms(sec):
    sec = max(0, int(sec or 0))
    h, r = divmod(sec, 3600)
    m, s = divmod(r, 60)
    return ("%d:%02d:%02d" % (h, m, s)) if h else ("%d:%02d" % (m, s))

# 以服务端 course/list 返回为准：progress / learnState / duration / learnDuration / finishDate
COURSES = [
    {"name": "文献检索与利用",    "progress": 100, "learnState": 2, "duration": 7450,  "learnDuration": 7450,  "finishDate": "2026-09-06"},
    {"name": "学术道德与规范",    "progress": 100, "learnState": 2, "duration": 8100,  "learnDuration": 8100,  "finishDate": "2026-09-06"},
    {"name": "科研方法概论",      "progress": 100, "learnState": 2, "duration": 9200,  "learnDuration": 9200,  "finishDate": "2026-09-06"},
    {"name": "数据可视化基础",    "progress": 100, "learnState": 2, "duration": 6300,  "learnDuration": 6300,  "finishDate": "2026-09-06"},
    {"name": "工程伦理导论",      "progress": 100, "learnState": 2, "duration": 11800, "learnDuration": 11800, "finishDate": "2026-09-06"},
    {"name": "学术论文写作",      "progress": 100, "learnState": 2, "duration": 7800,  "learnDuration": 7800,  "finishDate": "2026-09-06"},
]

total = sum(c["duration"] for c in COURSES)
learned = sum(c["learnDuration"] for c in COURSES)
done = sum(1 for c in COURSES if c["learnState"] == 2)
print("%-14s %-8s %-16s %s" % ("课程", "状态", "已看/总时", "完成日期"))
for c in COURSES:
    st = "已学完" if c["learnState"] == 2 else ("%d%%" % c["progress"])
    print("%-14s %-8s %-16s %s" % (c["name"], st, fmt_hms(c["learnDuration"]) + "/" + fmt_hms(c["duration"]), c["finishDate"]))
print()
print("共 %d 门课，全部完成 = %s；课内总时长 ~%s，已学 ~%s" % (len(COURSES), done == len(COURSES), fmt_hms(total), fmt_hms(learned)))


课程             状态       已看/总时            完成日期
文献检索与利用        已学完      2:04:10/2:04:10  2026-09-06
学术道德与规范        已学完      2:15:00/2:15:00  2026-09-06
科研方法概论         已学完      2:33:20/2:33:20  2026-09-06
数据可视化基础        已学完      1:45:00/1:45:00  2026-09-06
工程伦理导论         已学完      3:16:40/3:16:40  2026-09-06
学术论文写作         已学完      2:10:00/2:10:00  2026-09-06

共 6 门课，全部完成 = True；课内总时长 ~14:04:10，已学 ~14:04:10


## 4) 运行结果快照

In [1]:
import json

# 示意：主程序运行时的心跳 / 进度快照（账号已打码）
summary = {
    "account": "已打码（不显示真实账号）",
    "courses_total": 6,
    "courses_done": 6,
    "in_loop_review": True,
    "watch_total_sec": 16 * 3600 + 50 * 60 + 6,
    "cert_requirement_hours": 15,
}

done = summary["courses_done"] == summary["courses_total"]
print("账号:", summary["account"])
print("课程完成: %d/%d -> %s" % (summary["courses_done"], summary["courses_total"], "全部完成" if done else "未完成"))
print("累计学习时长:", fmt_hms(summary["watch_total_sec"]), "(证书要求 %d 小时)" % summary["cert_requirement_hours"])
print("是否进入循环补学:", summary["in_loop_review"])
print()
print("说明: 工具以服务端为准判定进度；学到期后进入循环补学，不再强制停表。")
print("证书是否下发取决于平台审核，脚本只负责把观看时长跑满。")


账号: 已打码（不显示真实账号）
课程完成: 6/6 -> 全部完成
累计学习时长: 16:50:06 (证书要求 15 小时)
是否进入循环补学: True

说明: 工具以服务端为准判定进度；学到期后进入循环补学，不再强制停表。
证书是否下发取决于平台审核，脚本只负责把观看时长跑满。


## 5) 使用方法

1. `pip install -r requirements.txt`
2. `playwright install chromium`
3. 复制 `账号密码.example.md` 为 `账号密码.md` 并填入自己的账号密码
4. `python 刷网课.py`

## 文件结构

| 文件 | 说明 |
|---|---|
| `刷网课.py` | 主程序（GUI + 挂机逻辑） |
| `requirements.txt` | 依赖 |
| `demo.ipynb` | 本展示文件 |
| `账号密码.example.md` | 账号密码占位模板 |
| `.gitignore` | 排除凭据与生成产物 |

---

> ⚠️ 仅供学习自动化研究，请遵守平台规则。请勿提交真实账号密码。